In [ ]:
import os

os.makedirs("../outputs_arabert", exist_ok=True)

print("✅ Dossier outputs_arabert/ créé !")
print(f"Chemin : {os.path.abspath('../outputs_arabert')}")

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

print("✅ Dataset ArSarcasm-v2 chargé !")
print(f"Train : {df_train.shape}")
print(f"Test  : {df_test.shape}")

In [ ]:
#Charger AraBERT et vérifier la tokenisation des emojis

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

from transformers import AutoTokenizer, AutoModel

tokenizer_arabert = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
model_arabert = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv2")
model_arabert.eval()

print("✅ AraBERT chargé !")

In [ ]:
!pip install "huggingface_hub<1.0" --quiet

In [ ]:
import subprocess
result = subprocess.run(["pip", "show", "huggingface_hub"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

from transformers import AutoTokenizer, AutoModel

tokenizer_arabert = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
model_arabert = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv2")
model_arabert.eval()

print("✅ AraBERT chargé !")

In [ ]:
exemple_avec_emoji = "صباحك جمال 🌿 🌹 🌿"
tokens_arabert = tokenizer_arabert.tokenize(exemple_avec_emoji)
print(f"Texte original : {exemple_avec_emoji}")
print(f"Tokens AraBERT : {tokens_arabert}")

encoded = tokenizer_arabert(exemple_avec_emoji, return_tensors="pt")
print(f"\nInput IDs : {encoded['input_ids']}")
decoded = tokenizer_arabert.decode(encoded['input_ids'][0])
print(f"Texte décodé (ce que le modèle voit vraiment) : {decoded}")

In [ ]:
#Vérifier MAX_LENGTH avec le tokenizer AraBERT

In [ ]:
longueurs_arabert = df_train["cleaned_tweet"].apply(
    lambda t: len(tokenizer_arabert.tokenize(t)) if isinstance(t, str) else 0
)
print(f"Longueur moyenne : {longueurs_arabert.mean():.1f} tokens")
print(f"95e percentile   : {longueurs_arabert.quantile(0.95):.1f} tokens")
print(f"99e percentile   : {longueurs_arabert.quantile(0.99):.1f} tokens")

In [ ]:
# Recharger le dataset ArSarcasm-v2 et nettoyer le texte

In [ ]:
import pandas as pd
import re
import pyarabic.araby as ar

df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

def clean_tweet(tweet):
    tweet = re.sub(r'http\S+|www.\S+', '', tweet)
    tweet = re.sub(r'@\w+|#\w+', '', tweet)
    tweet = re.sub(r'[^ء-ي\s]', '', tweet)
    tweet = ar.normalize_hamza(tweet)
    tweet = ar.normalize_ligature(tweet)
    tweet = ar.strip_tashkeel(tweet)
    return tweet.strip()

df_train["cleaned_tweet"] = df_train["tweet"].apply(clean_tweet)
df_test["cleaned_tweet"] = df_test["tweet"].apply(clean_tweet)

print("✅ Dataset ArSarcasm-v2 rechargé et nettoyé !")
print(f"Train : {df_train.shape}")
print(f"Test  : {df_test.shape}")

In [ ]:
longueurs_arabert = df_train["cleaned_tweet"].apply(
    lambda t: len(tokenizer_arabert.tokenize(t)) if isinstance(t, str) else 0
)
print(f"Longueur moyenne : {longueurs_arabert.mean():.1f} tokens")
print(f"95e percentile   : {longueurs_arabert.quantile(0.95):.1f} tokens")
print(f"99e percentile   : {longueurs_arabert.quantile(0.99):.1f} tokens")

#Extraction avec la méthode memmap 

In [ ]:
import numpy as np
import torch
from tqdm import tqdm

MAX_LENGTH = 64

def get_embedding_sequence_arabert(text):
    if not isinstance(text, str):
        text = ""
    inputs = tokenizer_arabert(text, return_tensors="pt",
                                truncation=True, max_length=MAX_LENGTH,
                                padding="max_length")
    with torch.no_grad():
        outputs = model_arabert(**inputs)
    sequence = outputs.last_hidden_state.squeeze(0).numpy()
    return sequence

N_train = len(df_train)
N_test = len(df_test)
seq_shape = (MAX_LENGTH, 768)

print(f"⏳ Extraction + sauvegarde directe train ({N_train} tweets)...")

train_mmap = np.lib.format.open_memmap(
    "../outputs_arabert/train_embeddings_sequence.npy",
    mode='w+', dtype=np.float32, shape=(N_train,) + seq_shape
)

textes_train = df_train["cleaned_tweet"].tolist()
for i in tqdm(range(N_train)):
    train_mmap[i] = get_embedding_sequence_arabert(textes_train[i])

train_mmap.flush()
del train_mmap

print("✅ Train embeddings sauvegardés !")

In [ ]:
#Extraction test

In [ ]:
print(f"⏳ Extraction + sauvegarde directe test ({N_test} tweets)...")

test_mmap = np.lib.format.open_memmap(
    "../outputs_arabert/test_embeddings_sequence.npy",
    mode='w+', dtype=np.float32, shape=(N_test,) + seq_shape
)

textes_test = df_test["cleaned_tweet"].tolist()
for i in tqdm(range(N_test)):
    test_mmap[i] = get_embedding_sequence_arabert(textes_test[i])

test_mmap.flush()
del test_mmap

print("✅ Test embeddings sauvegardés !")

np.save("../outputs_arabert/train_labels.npy", df_train["sarcasm"].astype(int).values)
np.save("../outputs_arabert/test_labels.npy", df_test["sarcasm"].astype(int).values)

print("✅ Tous les fichiers sauvegardés dans outputs_arabert/ !")

In [ ]:
#le chargement des features pragmatiques

In [ ]:
import numpy as np
import gc
gc.collect()

X_train_seq_arabert = np.load("../outputs_arabert/train_embeddings_sequence.npy", mmap_mode='r')
X_test_seq_arabert = np.load("../outputs_arabert/test_embeddings_sequence.npy", mmap_mode='r')

print("✅ Séquences AraBERT rechargées en mode memmap !")
print(f"Train : {X_train_seq_arabert.shape}")
print(f"Test  : {X_test_seq_arabert.shape}")

# Réutiliser les features pragmatiques déjà extraites (18 features, même texte)
train_prag_v2_full = pd.read_csv("../outputs/train_pragmatic_v2.csv")
test_prag_v2_full = pd.read_csv("../outputs/test_pragmatic_v2.csv")

X_train_prag_arabert = train_prag_v2_full.drop("label", axis=1).values
X_test_prag_arabert = test_prag_v2_full.drop("label", axis=1).values

y_train_full_arabert = np.load("../outputs_arabert/train_labels.npy")
y_test_arabert = np.load("../outputs_arabert/test_labels.npy")

from sklearn.model_selection import train_test_split

indices_arabert = np.arange(len(y_train_full_arabert))

idx_train_arabert, idx_val_arabert = train_test_split(
    indices_arabert,
    test_size=0.15,
    stratify=y_train_full_arabert,
    random_state=42
)

print(f"\n✅ Split train/val créé !")
print(f"Train final : {len(idx_train_arabert)} tweets")
print(f"Validation  : {len(idx_val_arabert)} tweets")
print(f"Test        : {len(y_test_arabert)} tweets")

In [ ]:
#architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report, precision_score, recall_score, accuracy_score
from scipy.stats import wilcoxon

class SoftAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SoftAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_output):
        scores = self.attention(lstm_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class SAttBLSTMConvNet(nn.Module):
    def __init__(self, embed_dim=768, conv_channels=128, lstm_hidden=128, prag_dim=18, dropout=0.3):
        super(SAttBLSTMConvNet, self).__init__()
        self.conv1 = nn.Conv1d(embed_dim, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(conv_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = SoftAttention(lstm_hidden * 2)
        self.dropout_final = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden * 2 + prag_dim, 128)
        self.fc2 = nn.Linear(128, 2)
    def forward(self, x_seq, x_prag):
        x = x_seq.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        combined = torch.cat([context, x_prag], dim=1)
        combined = self.dropout_final(combined)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class SarcasmDatasetMemmap(Dataset):
    def __init__(self, X_seq_memmap, X_prag, y, indices):
        self.X_seq_memmap = X_seq_memmap
        self.X_prag = torch.FloatTensor(X_prag[indices])
        self.y = torch.LongTensor(y[indices])
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        seq = np.array(self.X_seq_memmap[self.indices[idx]])
        return torch.FloatTensor(seq), self.X_prag[idx], self.y[idx]

def entrainer_avec_early_stopping_memmap(X_seq_memmap, X_prag, y, idx_train, idx_val,
                                           lr=0.001, max_epochs=50, patience=10, batch_size=32):
    train_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    class_counts = np.bincount(y[idx_train])
    weights = torch.FloatTensor(1.0 / class_counts)
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=X_prag.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    best_f1 = 0
    best_model_state = None
    epochs_sans_amelioration = 0
    for epoch in range(max_epochs):
        model.train()
        total_loss = 0
        for x_seq_b, x_prag_b, y_b in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq_b, x_prag_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x_seq_b, x_prag_b, y_b in val_loader:
                outputs = model(x_seq_b, x_prag_b)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.numpy())
                all_labels.extend(y_b.numpy())
        f1_val = f1_score(all_labels, all_preds, average='macro')
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1}/{max_epochs} - Loss: {total_loss/len(train_loader):.4f} - F1 val: {f1_val:.4f}")
        if f1_val > best_f1:
            best_f1 = f1_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= patience:
            print(f"   ⏹️ Early stopping à l'époque {epoch+1}")
            break
    model.load_state_dict(best_model_state)
    print(f"✅ Meilleur F1 validation atteint : {best_f1:.4f}")
    return model

def evaluer_test_complet_memmap(model, X_seq_memmap, X_prag, y, idx_test, batch_size=32):
    test_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq_b, x_prag_b, y_b in test_loader:
            outputs = model(x_seq_b, x_prag_b)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_b.numpy())
    resultats = {
        "F1_macro": f1_score(all_labels, all_preds, average='macro'),
        "Precision_macro": precision_score(all_labels, all_preds, average='macro'),
        "Recall_macro": recall_score(all_labels, all_preds, average='macro'),
        "Accuracy": accuracy_score(all_labels, all_preds),
        "MCC": matthews_corrcoef(all_labels, all_preds),
        "F1_sarcastique": f1_score(all_labels, all_preds, pos_label=1),
        "Precision_sarcastique": precision_score(all_labels, all_preds, pos_label=1),
        "Recall_sarcastique": recall_score(all_labels, all_preds, pos_label=1)
    }
    print(classification_report(all_labels, all_preds))
    print(f"\n{'='*50}\nRÉSUMÉ DES MÉTRIQUES\n{'='*50}")
    for nom, valeur in resultats.items():
        print(f"{nom:25} : {valeur:.4f}")
    return resultats

print("✅ Architecture et fonctions (AraBERT) prêtes !")

In [ ]:
def evaluer_hyperparam_arabert(lr, epochs=5, batch_size=32):
    torch.manual_seed(42)
    
    train_dataset = SarcasmDatasetMemmap(X_train_seq_arabert, X_train_prag_arabert, y_train_full_arabert, idx_train_arabert)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = SarcasmDatasetMemmap(X_train_seq_arabert, X_train_prag_arabert, y_train_full_arabert, idx_val_arabert)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    class_counts = np.bincount(y_train_full_arabert[idx_train_arabert])
    weights = torch.FloatTensor(1.0 / class_counts)
    
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=18)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x_seq, x_prag, y in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq, x_prag)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq, x_prag, y in val_loader:
            outputs = model(x_seq, x_prag)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y.numpy())
    
    f1_val = f1_score(all_labels, all_preds, average='macro')
    return f1_val, total_loss / len(train_loader)


print("⏳ Test de plusieurs learning rates sur le validation set (AraBERT)...\n")
learning_rates = [0.001, 0.0001, 0.00001]
resultats_hyperparam_arabert = []

for lr in learning_rates:
    print(f"⏳ Test avec lr={lr}...")
    f1_val, loss_final = evaluer_hyperparam_arabert(lr, epochs=5)
    print(f"   F1 macro (validation) = {f1_val:.4f} | Loss finale = {loss_final:.4f}\n")
    resultats_hyperparam_arabert.append({"learning_rate": lr, "F1_validation": f1_val, "loss": loss_final})

hyperparam_arabert_df = pd.DataFrame(resultats_hyperparam_arabert)
hyperparam_arabert_df = hyperparam_arabert_df.sort_values("F1_validation", ascending=False)

print("="*50)
print("RÉSULTATS — Comparaison des learning rates (AraBERT)")
print("="*50)
print(hyperparam_arabert_df.to_string(index=False))

meilleur_lr_arabert = hyperparam_arabert_df.iloc[0]

In [ ]:
# l'entraînement final — 5 runs SANS features (avec lr=0.0001)

In [ ]:
seeds = [42, 123, 456, 789, 101]

resultats_5runs_sans_arabert = []

X_train_prag_zeros_arabert = np.zeros_like(X_train_prag_arabert)
X_test_prag_zeros_arabert = np.zeros_like(X_test_prag_arabert)

print("="*60)
print("5 RUNS — SANS features pragmatiques (ArSarcasm-v2, AraBERT)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_s = entrainer_avec_early_stopping_memmap(
        X_train_seq_arabert, X_train_prag_zeros_arabert, y_train_full_arabert,
        idx_train_arabert, idx_val_arabert,
        lr=0.0001, max_epochs=50, patience=10
    )
    
    idx_test_all_arabert = np.arange(len(y_test_arabert))
    resultats_metriques = evaluer_test_complet_memmap(
        model_s, X_test_seq_arabert, X_test_prag_zeros_arabert, y_test_arabert, idx_test_all_arabert
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_sans_arabert.append(resultats_metriques)

print("\n✅ 5 runs SANS features (AraBERT) terminés !")
pd.DataFrame(resultats_5runs_sans_arabert).to_csv("../outputs_arabert/5runs_sans_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
seeds = [42, 123, 456, 789, 101]

resultats_5runs_sans_arabert = []

X_train_prag_zeros_arabert = np.zeros_like(X_train_prag_arabert)
X_test_prag_zeros_arabert = np.zeros_like(X_test_prag_arabert)

print("="*60)
print("5 RUNS — SANS features pragmatiques (ArSarcasm-v2, AraBERT)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_s = entrainer_avec_early_stopping_memmap(
        X_train_seq_arabert, X_train_prag_zeros_arabert, y_train_full_arabert,
        idx_train_arabert, idx_val_arabert,
        lr=0.0001, max_epochs=50, patience=10
    )
    
    idx_test_all_arabert = np.arange(len(y_test_arabert))
    resultats_metriques = evaluer_test_complet_memmap(
        model_s, X_test_seq_arabert, X_test_prag_zeros_arabert, y_test_arabert, idx_test_all_arabert
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_sans_arabert.append(resultats_metriques)

print("\n✅ 5 runs SANS features (AraBERT) terminés !")
pd.DataFrame(resultats_5runs_sans_arabert).to_csv("../outputs_arabert/5runs_sans_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report, precision_score, recall_score, accuracy_score
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
import gc

print("✅ Imports de base chargés !")

In [ ]:
gc.collect()

X_train_seq_arabert = np.load("../outputs_arabert/train_embeddings_sequence.npy", mmap_mode='r')
X_test_seq_arabert = np.load("../outputs_arabert/test_embeddings_sequence.npy", mmap_mode='r')

train_prag_v2_full = pd.read_csv("../outputs/train_pragmatic_v2.csv")
test_prag_v2_full = pd.read_csv("../outputs/test_pragmatic_v2.csv")

X_train_prag_arabert = train_prag_v2_full.drop("label", axis=1).values
X_test_prag_arabert = test_prag_v2_full.drop("label", axis=1).values

y_train_full_arabert = np.load("../outputs_arabert/train_labels.npy")
y_test_arabert = np.load("../outputs_arabert/test_labels.npy")

indices_arabert = np.arange(len(y_train_full_arabert))
idx_train_arabert, idx_val_arabert = train_test_split(
    indices_arabert, test_size=0.15, stratify=y_train_full_arabert, random_state=42
)

print("✅ Données AraBERT rechargées !")
print(f"Train seq : {X_train_seq_arabert.shape}")
print(f"Test seq  : {X_test_seq_arabert.shape}")
print(f"Train final : {len(idx_train_arabert)} | Val : {len(idx_val_arabert)}")

In [ ]:
class SoftAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SoftAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_output):
        scores = self.attention(lstm_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class SAttBLSTMConvNet(nn.Module):
    def __init__(self, embed_dim=768, conv_channels=128, lstm_hidden=128, prag_dim=18, dropout=0.3):
        super(SAttBLSTMConvNet, self).__init__()
        self.conv1 = nn.Conv1d(embed_dim, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(conv_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = SoftAttention(lstm_hidden * 2)
        self.dropout_final = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden * 2 + prag_dim, 128)
        self.fc2 = nn.Linear(128, 2)
    def forward(self, x_seq, x_prag):
        x = x_seq.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        combined = torch.cat([context, x_prag], dim=1)
        combined = self.dropout_final(combined)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class SarcasmDatasetMemmap(Dataset):
    def __init__(self, X_seq_memmap, X_prag, y, indices):
        self.X_seq_memmap = X_seq_memmap
        self.X_prag = torch.FloatTensor(X_prag[indices])
        self.y = torch.LongTensor(y[indices])
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        seq = np.array(self.X_seq_memmap[self.indices[idx]])
        return torch.FloatTensor(seq), self.X_prag[idx], self.y[idx]

def entrainer_avec_early_stopping_memmap(X_seq_memmap, X_prag, y, idx_train, idx_val,
                                           lr=0.001, max_epochs=50, patience=10, batch_size=32):
    train_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    class_counts = np.bincount(y[idx_train])
    weights = torch.FloatTensor(1.0 / class_counts)
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=X_prag.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    best_f1 = 0
    best_model_state = None
    epochs_sans_amelioration = 0
    for epoch in range(max_epochs):
        model.train()
        total_loss = 0
        for x_seq_b, x_prag_b, y_b in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq_b, x_prag_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x_seq_b, x_prag_b, y_b in val_loader:
                outputs = model(x_seq_b, x_prag_b)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.numpy())
                all_labels.extend(y_b.numpy())
        f1_val = f1_score(all_labels, all_preds, average='macro')
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1}/{max_epochs} - Loss: {total_loss/len(train_loader):.4f} - F1 val: {f1_val:.4f}")
        if f1_val > best_f1:
            best_f1 = f1_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= patience:
            print(f"   ⏹️ Early stopping à l'époque {epoch+1}")
            break
    model.load_state_dict(best_model_state)
    print(f"✅ Meilleur F1 validation atteint : {best_f1:.4f}")
    return model

def evaluer_test_complet_memmap(model, X_seq_memmap, X_prag, y, idx_test, batch_size=32):
    test_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq_b, x_prag_b, y_b in test_loader:
            outputs = model(x_seq_b, x_prag_b)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_b.numpy())
    resultats = {
        "F1_macro": f1_score(all_labels, all_preds, average='macro'),
        "Precision_macro": precision_score(all_labels, all_preds, average='macro'),
        "Recall_macro": recall_score(all_labels, all_preds, average='macro'),
        "Accuracy": accuracy_score(all_labels, all_preds),
        "MCC": matthews_corrcoef(all_labels, all_preds),
        "F1_sarcastique": f1_score(all_labels, all_preds, pos_label=1),
        "Precision_sarcastique": precision_score(all_labels, all_preds, pos_label=1),
        "Recall_sarcastique": recall_score(all_labels, all_preds, pos_label=1)
    }
    print(classification_report(all_labels, all_preds))
    print(f"\n{'='*50}\nRÉSUMÉ DES MÉTRIQUES\n{'='*50}")
    for nom, valeur in resultats.items():
        print(f"{nom:25} : {valeur:.4f}")
    return resultats

print("✅ Architecture et fonctions redéfinies !")

In [ ]:
seeds = [42, 123, 456, 789, 101]

resultats_5runs_sans_arabert = []

X_train_prag_zeros_arabert = np.zeros_like(X_train_prag_arabert)
X_test_prag_zeros_arabert = np.zeros_like(X_test_prag_arabert)

print("="*60)
print("5 RUNS — SANS features pragmatiques (ArSarcasm-v2, AraBERT)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_s = entrainer_avec_early_stopping_memmap(
        X_train_seq_arabert, X_train_prag_zeros_arabert, y_train_full_arabert,
        idx_train_arabert, idx_val_arabert,
        lr=0.0001, max_epochs=50, patience=10
    )
    
    idx_test_all_arabert = np.arange(len(y_test_arabert))
    resultats_metriques = evaluer_test_complet_memmap(
        model_s, X_test_seq_arabert, X_test_prag_zeros_arabert, y_test_arabert, idx_test_all_arabert
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_sans_arabert.append(resultats_metriques)

print("\n✅ 5 runs SANS features (AraBERT) terminés !")
pd.DataFrame(resultats_5runs_sans_arabert).to_csv("../outputs_arabert/5runs_sans_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
#5 runs AVEC features pragmatiques

In [ ]:
resultats_5runs_avec_arabert = []

print("="*60)
print("5 RUNS — AVEC features pragmatiques (ArSarcasm-v2, AraBERT)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_a = entrainer_avec_early_stopping_memmap(
        X_train_seq_arabert, X_train_prag_arabert, y_train_full_arabert,
        idx_train_arabert, idx_val_arabert,
        lr=0.0001, max_epochs=50, patience=10
    )
    
    idx_test_all_arabert = np.arange(len(y_test_arabert))
    resultats_metriques = evaluer_test_complet_memmap(
        model_a, X_test_seq_arabert, X_test_prag_arabert, y_test_arabert, idx_test_all_arabert
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_avec_arabert.append(resultats_metriques)

print("\n✅ 5 runs AVEC features (AraBERT) terminés !")
pd.DataFrame(resultats_5runs_avec_arabert).to_csv("../outputs_arabert/5runs_avec_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
#Test de Wilcoxon final pour AraBERT

In [ ]:
f1_sans_arabert = [r["F1_macro"] for r in resultats_5runs_sans_arabert]
f1_avec_arabert = [r["F1_macro"] for r in resultats_5runs_avec_arabert]

print("F1 SANS (AraBERT) :", [f"{f:.4f}" for f in f1_sans_arabert])
print("F1 AVEC (AraBERT) :", [f"{f:.4f}" for f in f1_avec_arabert])

moyenne_sans = np.mean(f1_sans_arabert)
std_sans = np.std(f1_sans_arabert)
moyenne_avec = np.mean(f1_avec_arabert)
std_avec = np.std(f1_avec_arabert)

print(f"\nSANS features : {moyenne_sans:.4f} ± {std_sans:.4f}")
print(f"AVEC features : {moyenne_avec:.4f} ± {std_avec:.4f}")

stat, p_value = wilcoxon(f1_avec_arabert, f1_sans_arabert, alternative='greater')
print(f"\nWilcoxon unilatéral (H1: AVEC > SANS) — AraBERT")
print(f"Statistique : {stat:.4f} | P-value : {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ Résultat SIGNIFICATIF (p < 0.05) !")
elif p_value < 0.10:
    print("\n⚠️ Résultat MARGINALEMENT significatif (p < 0.10)")
else:
    print("\n⚠️ Résultat NON significatif")

In [ ]:
import pandas as pd

fichiers = {
    "MARBERT": "../outputs/5runs_sans_features_complet.csv",
    "AraBERT": "../outputs_arabert/5runs_sans_features_complet.csv",
    "CamelBERT": "../outputs_camelbert/5runs_sans_features_complet.csv"
}

metriques = ["F1_macro", "Precision_macro", "Recall_macro", "Accuracy", "MCC"]

print("="*90)
print("RÉSULTATS SANS FEATURES — ArSarcasm-v2 (moyenne ± écart-type sur 5 seeds)")
print("="*90)

for modele, chemin in fichiers.items():
    df = pd.read_csv(chemin)
    print(f"\n--- {modele} ---")
    for m in metriques:
        if m in df.columns:
            moyenne = df[m].mean()
            ecart_type = df[m].std()
            print(f"  {m:20} : {moyenne:.4f} ± {ecart_type:.4f}")

In [ ]:
fichiers_avec = {
    "MARBERT": "../outputs/5runs_avec_features_complet.csv",
    "AraBERT": "../outputs_arabert/5runs_avec_features_complet.csv",
    "CamelBERT": "../outputs_camelbert/5runs_avec_features_complet.csv"
}

print("="*90)
print("RÉSULTATS AVEC FEATURES — ArSarcasm-v2 (moyenne ± écart-type sur 5 seeds)")
print("="*90)

for modele, chemin in fichiers_avec.items():
    df = pd.read_csv(chemin)
    print(f"\n--- {modele} ---")
    for m in metriques:
        if m in df.columns:
            moyenne = df[m].mean()
            ecart_type = df[m].std()
            print(f"  {m:20} : {moyenne:.4f} ± {ecart_type:.4f}")

In [ ]:
import pandas as pd

fichiers_moji_sans = {
    "MARBERT": "../outputs_dedup_correction/5runs_sans_features_complet.csv",
    "AraBERT": "../outputs_dedup_arabert_correction/5runs_sans_features_complet.csv",
    "CamelBERT": "../outputs_dedup_camelbert_correction/5runs_sans_features_complet.csv"
}

metriques = ["F1_macro", "Precision_macro", "Recall_macro", "Accuracy", "MCC"]

print("="*90)
print("RÉSULTATS SANS FEATURES — ArSarcaMoji corrigé (moyenne ± écart-type)")
print("="*90)

for modele, chemin in fichiers_moji_sans.items():
    df = pd.read_csv(chemin)
    print(f"\n--- {modele} (n={len(df)} runs) ---")
    for m in metriques:
        if m in df.columns:
            moyenne = df[m].mean()
            ecart_type = df[m].std()
            print(f"  {m:20} : {moyenne:.4f} ± {ecart_type:.4f}")